### Initialize the Environment:

##### Virtual Environment Commands

| Command | Linux/Mac | GitBash |
| ------- | --------- | ------- |
| Create | `python3 -m venv venv` | `python -m venv venv` |
| Activate | `source venv/bin/activate` | `source venv/Scripts/activate` |
| Install | `pip install -r requirements.txt` | `pip install -r requirements.txt` |
| Deactivate | `deactivate` | `deactivate` |

##### Select the Kernel (This will be in the Requirements.txt eventually)

Using the venv (Python 3.13.2) located  in venv/bin/python)


### **Project Overview & Plan**

Capstone Project for Code:You Data Analysis track. This project analyzes Beer Recipes for frequency of uploads for various beer styles, while capturing preferences of strength, hopiness and batch size.    The goal of the project is to demonstrate a general knowledge of Python (Pandas, Numpy, MatLibPlot, Plotly), SQL(MySQL), Tableu, Cursor and ChatGPT.

**Data Sources:**

The datasets used in this project are all related to online beer recipes. One dataset contains the different styles of beer as recognized by the Beer Judge Certification Program (BJCP.org).
- [Beersmith Recipes](https://beersmithrecipes.com/recent/) - scraped data that contains certain fields of 100% of the all grain beer recipes that have been uploaded by users.

- [Brewers Friend All-Grain Recipes](https://www.brewersfriend.com/homebrew-recipes/all-grain/) - scraped data that contains select fields of 100% of the all-grain beer recipes that have been uploaded by users.
- [Kaggle - Brewers Friend Recipes](https://www.kaggle.com/datasets/jtrofe/beer-recipes) - data from Kaggle that contains a subset of beer recipes.
- [BJCP - Judging Styles of Beer](https://github.com/ascholer/bjcp-styleview/blob/main/styles.json) - dataset that contains the criterea used to judge beer. Will help determine if recipes meet the criterea to be considered a specific style of beer.

In [21]:
import pandas as pd
import numpy as np
import re
# import matplotlib
from pandas import DataFrame
# import matplotlib.pyplot as plt
# from matplotlib.ticker import FuncFormatter
# from rich.console import Console
# from rich.table import Table

Note: styles.json from: https://github.com/ascholer/bjcp-styleview

In [22]:
bs_recipes = pd.read_csv('beersmith_recipes.csv')
bf_recipes = pd.read_csv('bf_recipes.csv')
kag_recipes = pd.read_csv('recipeData.csv', encoding='ISO-8859-1')
bjcp_styles = pd.read_json('styles.json')

In [23]:
print(bf_recipes.shape)
print(bs_recipes.shape)
print(kag_recipes.shape)
print(bjcp_styles.shape)

(215580, 8)
(63121, 5)
(73861, 23)
(116, 28)


In [24]:
# simplify dataframe names
bf_df = bf_recipes 
bs_df = bs_recipes
kag_df = kag_recipes
style_df = bjcp_styles

## Data Cleanup

### Beer Smith Recipes - beersmith_recipes.csv = bs_df
Will analyze data, resolve missing data, split the stats column into the individual components and prepare to join with Brewers Friend data. While cleaning up the data, will create a function to clean up the data in the other datasets. In the Beer Smith data there were 31 rows missing the Recipe Name. Since the name wasn't crucial to the analysis of the recipes, but the recipe information was still of value, we assigned a generic name to each of the rows missing the Recipe Name.

In [25]:
# print(bf_df.info())
print(bs_df.info())
# print(kag_df.info())
# print(style_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63121 entries, 0 to 63120
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Recipe Name  63090 non-null  object
 1   Recipe URL   63121 non-null  object
 2   Beer Style   63121 non-null  object
 3   Brewer       63121 non-null  object
 4   Stats        63121 non-null  object
dtypes: object(5)
memory usage: 2.4+ MB
None


In [26]:
# Create a copy for cleaning the dataframe
bs_df_cleaned = bs_df.copy()

# Count null values for each column
null_counts = bs_df_cleaned.isnull().sum()

# Filter columns with null counts greater than zero
columns_with_nulls = null_counts[null_counts > 0].index.tolist()

print(null_counts)
print(columns_with_nulls)


Recipe Name    31
Recipe URL      0
Beer Style      0
Brewer          0
Stats           0
dtype: int64
['Recipe Name']


In [27]:
"""
# Function to generate "No Name" with incrementing numbers
# This function was created with the direct help of Perplexity AI. I was unaware of the generator 
# function until I prompted Perplexity to help me write a function to replace missing data with "No Name" 
# and index it starting at 1 and increment by 1 until all null values have been replaced. 
# 
def generate_no_name():
    counter = 1
    while True:
        yield f"No Name {counter}"
        counter += 1

# Create a generator for "No Name" values
no_name_gen = generate_no_name()

# Replace null values with "No Name" followed by a unique number
for column in columns_with_nulls:
    mask = bs_df_cleaned[column].isnull()
    bs_df_cleaned.loc[mask, column] = [next(no_name_gen) for _ in range(mask.sum())]

# Print the null value counts after replacement
print(f"There were: {null_counts} missing recipes replaced with No Name followed by an incremented number ")
print(bs_df_cleaned.isnull().sum()[columns_with_nulls])
"""

'\n# Function to generate "No Name" with incrementing numbers\n# This function was created with the direct help of Perplexity AI. I was unaware of the generator \n# function until I prompted Perplexity to help me write a function to replace missing data with "No Name" \n# and index it starting at 1 and increment by 1 until all null values have been replaced. \n# \ndef generate_no_name():\n    counter = 1\n    while True:\n        yield f"No Name {counter}"\n        counter += 1\n\n# Create a generator for "No Name" values\nno_name_gen = generate_no_name()\n\n# Replace null values with "No Name" followed by a unique number\nfor column in columns_with_nulls:\n    mask = bs_df_cleaned[column].isnull()\n    bs_df_cleaned.loc[mask, column] = [next(no_name_gen) for _ in range(mask.sum())]\n\n# Print the null value counts after replacement\nprint(f"There were: {null_counts} missing recipes replaced with No Name followed by an incremented number ")\nprint(bs_df_cleaned.isnull().sum()[columns_w

In [20]:
# Function to generate "No Name" with incrementing numbers

# This function was created with the direct help of Perplexity AI. I was unaware of the generator 
# function until I prompted Perplexity to help me write a function to replace missing data with "No Name" 
# and index it starting at 1 and increment by 1 until all null values have been replaced. If I have time
# I will revisit to have the program prompt what missing data to replace by column. 

def generate_no_name():
    counter = 1
    while True:
        yield f"No Name {counter}"
        counter += 1

# Create a generator for "No Name" values
no_name_gen = generate_no_name()

# Dictionary to store the number of replacements for each column
replacements_count = {}

# Dictionary to store an example of a changed row for each column
changed_row_examples = {}

# Replace null values with "No Name" followed by a unique number
for column in columns_with_nulls:
    mask = bs_df_cleaned[column].isnull()
    num_replacements = mask.sum()
    
    if num_replacements > 0:
        replacements = [next(no_name_gen) for _ in range(num_replacements)]
        bs_df_cleaned.loc[mask, column] = replacements
        replacements_count[column] = num_replacements
        
        # Store an example of a changed row
        changed_row_index = mask.idxmax()
        changed_row_examples[column] = bs_df_cleaned.loc[changed_row_index]

# Print the number of replacements made for each column
print("Number of replacements made:")
for column, count in replacements_count.items():
    print(f"{column}: {count}")

# Print the shape of the updated dataframe
print(f"\nShape of bs_df_cleaned: {bs_df_cleaned.shape}")

# Display an example of a changed row for each modified column
print("\nExamples of rows with changed data:")
for column, row in changed_row_examples.items():
    print(f"\nColumn: {column}")
    print(row)


Number of replacements made:
Recipe Name: 31

Shape of bs_df_cleaned: (63121, 5)

Examples of rows with changed data:

Column: Recipe Name
Recipe Name                                            No Name 1
Recipe URL     https://beersmithrecipes.com/viewrecipe/4987166/-
Beer Style                                     Best Bitter (11B)
Brewer                                                   Theossi
Stats          OG: 1.042 (10.5° P), Bitterness: 30.4 IBUs, AB...
Name: 595, dtype: object


### Splitting the Stats & Beer Style columns into individual columns of Data: 
Stats "OG: 1.073 (17.7° P), Bitterness: 34.5 IBUs, ABV: 6.9 %"
Beer Style "Dark Mild (13A)"

| OG| Plato| IBU's| ABV|Calc  FG |
|----------:|----------:|----------:|----------:|----------:|
| 1.073| 17.7| 34.5 | 6.9 |Calc|

| Style|Number|
|----------:|----------:|
| Dark Mild| 13A| 

The Final Gravity will be calculated using the Balling formula: 


### Balling Formula for Final Gravity (FG) Calculation (ChatGPT)
To estimate the **Final Gravity (FG)** using the **Original Gravity (OG) in Plato (°P)** and the **Alcohol by Volume (ABV)**, use the **Balling formula**:

\[
FG = 1 - \frac{ABV}{2.0665} \times (1.001843 - 0.002318474 \times OG°P - 0.000007775 \times OG°P^2 - 0.000000034 \times OG°P^3)
\]

### Example Calculation
If **Original Gravity** is **12.5°P** and the **ABV** is **5.5%**, the estimated Final Gravity (FG) is:

\[
FG = 1 - \frac{5.5}{2.0665} \times (1.001843 - 0.002318474 \times 12.5 - 0.000007775 \times 12.5^2 - 0.000000034 \times 12.5^3)
\]

After solving, we get:

\[
FG \approx 1.0104
\]


In [ ]:
# Imported "re" Python module to assist with parsing the data with pattern matching
# Stats column contains "OG: 1.073 (17.7° P), Bitterness: 34.5 IBUs, ABV: 6.9 %"
# Beer Style column contains "Dark Mild (13A)"
# ""
def parse_stats_column(df: pd.DataFrame) -> pd.DataFrame:
    def extract_value(text: str, pattern: str) -> float:
        match = re.search(pattern, text)
        return float(match.group(1)) if match else None

    df['OG'] = df['Stats'].apply(lambda x: extract_value(x, r'OG: (\d+\.\d+)'))
    df['Plato'] = df['Stats'].apply(lambda x: extract_value(x, r'(\d+\.\d+)° P'))
    df['IBUs'] = df['Stats'].apply(lambda x: extract_value(x, r'Bitterness: (\d+\.\d+)'))
    df['ABV'] = df['Stats'].apply(lambda x: extract_value(x, r'ABV: (\d+\.\d+)'))

    return df

In [36]:
bs_df_cleaned = parse_stats_column(bs_df_cleaned)

def calculate_final_gravity(og_plato, abv):
    """
    Calculate Final Gravity (FG) using the Balling formula.

    Parameters:
    og_plato (float): Original gravity in degrees Plato.
    abv (float): Alcohol by volume percentage.

    Returns:
    float: Estimated Final Gravity (FG) in Specific Gravity (SG) units.
    """
    fg = 1 - (abv / 2.0665) * (1.001843 - 0.002318474 * og_plato - 
                               0.000007775 * (og_plato ** 2) - 
                               0.000000034 * (og_plato ** 3))
    return fg

# Example usage:
og_plato = 12.5  # Example OG in Plato
abv = 5.5  # Example ABV
fg = calculate_final_gravity(og_plato, abv)
print(f"Estimated Final Gravity: {fg:.4f}")


In [ ]:
def main() -> None:
    """
    The main function for the Beer Recipe Capstone.
    """
    # Load the data
    ky_jobs = pd.read_excel('2022-2032_Occupational_Outlook.xlsx')

    # Clean the data
    ky_jobs = clean_ky_jobs_data(ky_jobs)

    # Add a new category column
    ky_jobs = new_category_from_rows(ky_jobs)

    # Calculate and summarize new job information
    descriptive_statistics = summarize_new_job_info(ky_jobs)
    print(descriptive_statistics)

    # Calculate category statistics
    category_stats = calculate_category_stats(ky_jobs)

    # Display category statistics
    display_category_stats(category_stats)

    # Plot the top 5 categories in job growth percent
    plot_top_5_category_job_growth(ky_jobs)

    # Plot the top 5 categories in new jobs
    plot_top_5_category_new_jobs(ky_jobs)

if __name__ == "__main__":
    main()